# Table 1 experiments — Colab notebook

This notebook runs the four benchmark configurations (A–D) for Table 1 of the
MCC paper, entirely inside a Colab VM: it starts a local instance of the
engine, replays the 52-case gold benchmark against it with the harness, and
copies every result to your Google Drive.

**Read `docs/paper/table1-runbook.md` in the repository first** — it explains
*why* each step exists. This notebook only automates the *how*.

### Non-negotiable ground rules (from the runbook)

1. **One model for the whole grid.** The model id is pinned in the
   configuration cell below. If it ever changes, every configuration must be
   re-run.
2. **Never edit gold labels mid-experiment** (`bench/cases.jsonl` is frozen).
3. **The 15 holdout cases never inform calibration.** Configuration D is
   reported on them; nothing else touches them.
4. **Copy results to Drive immediately after each run** (the cells do this
   for you — don't skip them). The Colab VM is ephemeral: a disconnect
   loses everything not yet on Drive.
5. **One configuration per session is fine**; a configuration interrupted
   halfway must be re-run in full. Config C is the long one (3–5 h) — start
   it early in the day and keep the tab open. Colab Pro is strongly
   recommended for C and B.

### Before you start — one-time setup

Open the **Secrets panel** (key icon in the left sidebar) and add, with
*notebook access* enabled:

- `AWS_BEARER_TOKEN_BEDROCK` — the Bedrock API key (ask your supervisor;
  never paste it into a cell).
- `GITHUB_TOKEN` — a GitHub personal access token with read access to the
  repository (only needed if the repository is private).

## Cell 1 — Clone the repository and install dependencies

Clones the project, checks out the working branch, and installs the backend
requirements into the Colab runtime. Re-run this cell whenever you start a
fresh Colab session — the VM starts empty every time.

Expected output: Python ≥ 3.11, a successful clone, and a pip install that
ends without errors.

In [ ]:
!python --version

import os
from google.colab import userdata

REPO = "/content/VA-marketSurveillance"
BRANCH = "main"

try:
    _token = userdata.get("GITHUB_TOKEN")
except Exception:
    _token = None
_url = (f"https://{_token}@github.com/maragem/VA-marketSurveillance"
        if _token else "https://github.com/maragem/VA-marketSurveillance")

if not os.path.isdir(REPO):
    !git clone {_url} {REPO}
%cd {REPO}
!git checkout {BRANCH}
!git pull origin {BRANCH}
!pip -q install -r backend/requirements.txt httpx

## Cell 2 — Mount Google Drive

All results are copied under `MyDrive/mcc-table1/`, one subfolder per
configuration, plus a `runlog.txt` recording what each result stamp is.
Approve the Drive authorisation popup when it appears.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/mcc-table1"
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("results will be archived under:", DRIVE_ROOT)

## Cell 3 — Experiment settings and helper functions

This cell pins the **model id** (ground rule 1) and defines three helpers
used by every configuration:

- `start_server(...)` — launches a fresh engine instance on port 8000 with a
  throwaway `DATA_DIR`, configured for the requested configuration:
  credentials on/off (config A runs without any), the citation-validation
  experiment flag (config B only), and calibration rules blanked (A/B/C) or
  seeded from the repository (D). It waits for `/api/health` and prints the
  health status and the **number of active calibration rules — check this
  line every time**: it must be `0` for A/B/C and `> 0` for D.
- `save_results(config_name, note)` — moves everything in `bench/results/`
  to the Drive folder for that configuration and appends a `runlog.txt`
  line. Run it immediately after each harness run.
- `stop_server()` — stops the running instance (called automatically by
  `start_server`).

The backend's own log is written to `/content/server.log` — look there if
anything misbehaves.

In [ ]:
import json, os, shutil, subprocess, tempfile, time, urllib.request

MODEL_ID = "eu.anthropic.claude-opus-4-8"   # ONE model for the whole grid
AWS_REGION = "eu-west-1"

_server = None

def stop_server():
    global _server
    if _server and _server.poll() is None:
        _server.terminate()
        try:
            _server.wait(10)
        except subprocess.TimeoutExpired:
            _server.kill()
    _server = None

def start_server(with_llm=True, skip_validation=False, with_calibration=False):
    global _server
    stop_server()
    data_dir = tempfile.mkdtemp()
    env = dict(os.environ, DATA_DIR=data_dir)
    for var in ("ANTHROPIC_API_KEY", "AWS_BEARER_TOKEN_BEDROCK",
                "EXPERIMENT_SKIP_CITATION_VALIDATION"):
        env.pop(var, None)
    if with_llm:
        from google.colab import userdata
        env["AWS_BEARER_TOKEN_BEDROCK"] = userdata.get("AWS_BEARER_TOKEN_BEDROCK")
        env["AWS_REGION"] = AWS_REGION
        env["ANTHROPIC_MODEL"] = MODEL_ID
    if skip_validation:
        env["EXPERIMENT_SKIP_CITATION_VALIDATION"] = "1"
    if not with_calibration:
        # A pre-existing working file stops the store from seeding the
        # repository's rules — this is how A/B/C run calibration-free.
        with open(os.path.join(data_dir, "calibration.json"), "w") as f:
            json.dump({s: [] for s in ("GLOBAL", "AI_ACT", "GDPR", "DSA", "UCPD")}, f)
    _server = subprocess.Popen(
        ["python", "-m", "uvicorn", "app.main:app", "--port", "8000"],
        cwd=f"{REPO}/backend", env=env,
        stdout=open("/content/server.log", "ab"), stderr=subprocess.STDOUT)
    for _ in range(90):
        try:
            health = json.load(urllib.request.urlopen(
                "http://localhost:8000/api/health", timeout=2))
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("server did not come up — check /content/server.log")
    cal = json.load(urllib.request.urlopen("http://localhost:8000/api/calibration"))
    n_rules = sum(len(v) for v in cal.values() if isinstance(v, list))
    print("health:", json.dumps(health))
    print(f"active calibration rules: {n_rules}"
          f"  <-- must be 0 for A/B/C, >0 for D")
    return health

def save_results(config_name, note=""):
    src = f"{REPO}/bench/results"
    dest = f"{DRIVE_ROOT}/{config_name}"
    os.makedirs(dest, exist_ok=True)
    moved = []
    if os.path.isdir(src):
        for name in sorted(os.listdir(src)):
            shutil.move(os.path.join(src, name), os.path.join(dest, name))
            moved.append(name)
    if not moved:
        raise RuntimeError("no result files found — did the harness run?")
    with open(f"{DRIVE_ROOT}/runlog.txt", "a") as f:
        f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} | {config_name} | "
                f"model={MODEL_ID} | {note} | files: {', '.join(moved)}\n")
    print(f"archived to {dest}:")
    for name in moved:
        print("  ", name)

print("helpers ready — model pinned to", MODEL_ID)

## Configuration A — retrieval-only (no LLM)

**What it measures:** what GraphRAG retrieval alone contributes, without any
language model. The server is started *without credentials*, so the engine
falls back to its deterministic heuristic mode.

**Cost/time:** free (no LLM calls), minutes.

Check in the output of `start_server` that the health line shows the
retrieval-only/heuristic provider (no model), and `active calibration
rules: 0`.

In [ ]:
start_server(with_llm=False)
!cd {REPO} && python bench/run.py --base-url http://localhost:8000 --runs 3
save_results("config-A", note="retrieval-only, no LLM credentials, calibration blank")

## Configuration C — deployed default (run this before B)

**What it measures:** the full agentic pipeline exactly as deployed —
triage, parallel regulation specialists, synthesis, citation validation —
with **no calibration rules**.

**Two runs are needed:**

1. the full 52-case set × 3 repetitions (the main Table 1 row, including the
   stability metric), and
2. the same protocol restricted to the 15 held-out cases (`--holdout`) —
   this becomes the **baseline that configuration D is compared against**.

**Cost/time:** this is the long one — 156 + 45 analyses, budget 3–5 hours
total. Keep the tab open. Watch the first few cases scroll by: if you see
timeout or throttling errors, interrupt, and re-run the *whole* cell with
`--concurrency 2` added.

Check: health shows Bedrock + the pinned model; `active calibration rules: 0`.

In [ ]:
start_server(with_llm=True)
!cd {REPO} && python bench/run.py --base-url http://localhost:8000 --runs 3
save_results("config-C-full", note="default pipeline, 52 cases x3, calibration blank")

In [ ]:
# C on the holdout only — D's baseline. Reuses the server started above;
# if you are resuming in a fresh session, run start_server(with_llm=True) first.
!cd {REPO} && python bench/run.py --base-url http://localhost:8000 --runs 3 --holdout
save_results("config-C-holdout", note="default pipeline, 15 holdout cases x3, calibration blank")

## Configuration B — citation validation disabled (ablation)

**What it measures:** what the citation-validation layer contributes. The
server is started with `EXPERIMENT_SKIP_CITATION_VALIDATION=1`, so findings
that validation would normally remove (hallucinated provision ids,
cross-regulation citations) survive into the report. The scorecard's
**invalid citations** metric is the headline number for this row — it is 0
by construction for A/C/D.

**Never set this flag on a shared or production instance.** It exists only
for this experiment.

**Cost/time:** 156 analyses, 2–4 hours.

The verification snippet after the run checks the raw reports for the
`agentic-unvalidated` engine label, which proves the flag took effect. If it
warns, do **not** save — check `/content/server.log` and re-run.

In [ ]:
start_server(with_llm=True, skip_validation=True)
!cd {REPO} && python bench/run.py --base-url http://localhost:8000 --runs 3

import glob
_reports = sorted(glob.glob(f"{REPO}/bench/results/*-reports.jsonl"))
_txt = open(_reports[-1]).read() if _reports else ""
if "agentic-unvalidated" in _txt:
    print("OK: experiment flag confirmed (engine=agentic-unvalidated)")
    save_results("config-B", note="EXPERIMENT_SKIP_CITATION_VALIDATION=1, 52 cases x3")
else:
    print("WARNING: 'agentic-unvalidated' not found in raw reports — the flag may "
          "not have taken effect. Inspect /content/server.log and the reports "
          "before saving; do not archive an unverified B run.")

## Between C and B ... and D: the calibration rule set

Configuration D needs the **human-approved calibration rules** committed to
the repository as `knowledge/calibration.json`. This step happens *outside
Colab*, in the deployed application, and is normally done by the supervisor
(runbook §6.1):

1. Deployed app → Benchmark page → calibration panel → **Export**.
2. Check every rule is a general legal principle (no case ids, no brand
   names, no campaign facts).
3. Commit the export over `knowledge/calibration.json`.

Only proceed to D once that commit exists — the cell below pulls it. If
`git pull` brings in no change to `knowledge/calibration.json`, stop and ask
the supervisor.

## Configuration D — C + calibration, holdout only

**What it measures:** what benchmark-distilled calibration adds, evaluated
on the 15 held-out cases that never informed rule-writing. Compare this
scorecard against `config-C-holdout` — same cases, same protocol, so the
delta is attributable to calibration alone.

The server is started *with* calibration this time (the fresh `DATA_DIR`
seeds from the repository file). Check the `active calibration rules` line
in the output: it must be **greater than 0**, matching the number of
approved rules. If it shows 0, the calibration commit is missing — stop.

**Cost/time:** 45 analyses, ~1 hour.

In [ ]:
!cd {REPO} && git pull origin {BRANCH}
start_server(with_llm=True, with_calibration=True)
!cd {REPO} && python bench/run.py --base-url http://localhost:8000 --runs 3 --holdout
save_results("config-D", note="calibration active, 15 holdout cases x3")

## Final check — what goes to the supervisor

The cell below lists everything archived on Drive and prints the run log.
Verify you have five folders — `config-A`, `config-C-full`,
`config-C-holdout`, `config-B`, `config-D` — each containing, per run, a
`.md` scorecard, a `.json` aggregate, and a `-reports.jsonl` raw-output
file, and that `runlog.txt` has one line per archive.

Then share the `mcc-table1` Drive folder with your supervisor. The results
will be verified (engine labels, holdout case counts, reproduction of the
scores from the raw reports) and committed to the repository — you do not
need push access.

In [ ]:
import os
for root, dirs, files in os.walk(DRIVE_ROOT):
    depth = root[len(DRIVE_ROOT):].count(os.sep)
    print("  " * depth + os.path.basename(root) + "/")
    for name in sorted(files):
        print("  " * (depth + 1) + name)
print("\n--- runlog.txt ---")
log = os.path.join(DRIVE_ROOT, "runlog.txt")
print(open(log).read() if os.path.exists(log) else "(missing!)")

## Troubleshooting

- **Server never comes up** — `!tail -50 /content/server.log`. Usual causes:
  pip install didn't finish (re-run Cell 1) or a bad secret name.
- **Health shows the wrong model/provider** — the pinned `MODEL_ID` cell was
  edited or `ANTHROPIC_API_KEY` leaked into the environment; restart the
  runtime (Runtime → Restart) and re-run Cells 1–3.
- **403 Marketplace / model-access error from Bedrock** — the model isn't
  enabled for the AWS account in that region; ask the supervisor to enable
  it (AWS console → Bedrock → Model access). Nothing to fix in the notebook.
- **Throttling or per-case timeouts** — re-run the configuration with
  `--concurrency 2` (and/or `--timeout 900`) appended to the harness command.
  A configuration with failed cases is not reportable — always re-run it
  cleanly.
- **Colab disconnected mid-run** — the configuration must be re-run in full.
  Earlier configurations already on Drive are safe.
- **`active calibration rules` is wrong for the configuration** — for A/B/C
  it must be 0, for D it must match the approved set. `start_server` handles
  this via a fresh `DATA_DIR`; if it's wrong, restart the runtime and re-run
  from Cell 3.